# Welcome to D ONE! 
### As a consultant at D ONE, you are part of a team that transforms complex data environments into high-impact decision intelligence platforms.

In this first notebook of our 2-part workshop series, you will build the foundational Medallion Data Architecture. You will ingest multi-source raw operational feeds—client projects, consultant assignments, and logged project hours—landing them safely into Databricks Unity Catalog before cleansing and curating them.

A rough overview of what we will build is shown below:


<pre>
┌───────────────────────────────────────────────────────────────────────────────┐
│                         DONE INTERNAL GOVERNANCE                              │
│                         catalog · schema · object                             │
└───────────────────────────────────────────────────────────────────────────────┘
                                      │
                                      ▼
┌───────────────────────────────────────────────────────────────────────────────┐
│                           MEDALLION PIPELINE                                  │
│                                                                               │
│   SOURCES              BRONZE                    SILVER              GOLD     │
│                                                                               │
│  ┌──────────┐       ┌────────────────┐       ┌────────────────┐  ┌──────────┐ │
│  │   CSVs   │ ───▶  │ raw_consult... │ ───▶  │ silver_cons... │─▶│ gold_proj│ │
│  │ Parquet  │ ───▶  │ raw_project... │ ───▶  │ silver_proj... │─▶│ practice │ │
│  │   JSON   │ ───▶  │ raw_timeshe... │ ───▶  │ silver_time... │─▶│ metrics  │ │
│  └──────────┘       └────────────────┘       └────────────────┘  └──────────┘ │
│                                                                               │
└───────────────────────────────────────────────────────────────────────────────┘
</pre>

## Step 0: Environment Setup

We start by setting up dynamic parameters based on your email to isolate your workspace.

In [0]:
%run ../___setup


💡 You can access the variables above using the ' `{CATALOG_NAME}` ' command.

## Step 1: Create Unity Catalog Governance

In Unity Catalog, a Catalog represents the top-tier namespace boundary (catalog.schema.table). 

Create your dedicated catalog using the command `CREATE CATALOG IF NOT EXISTS`

> You only need to replace the ...... placeholders.


In [0]:
load_hint("notebook_1", "step_1")

In [0]:

# 1.1 Create the dedicated practice catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOG_NAME}`")

# 1.2 Set current catalog context
spark.sql(f"USE CATALOG `{CATALOG_NAME}`")

# Verify context
display(spark.sql("SELECT current_catalog()"))



## Step 2: Create Schema
A Schema logical container holds related tables, views, and volumes inside Unity Catalog.

Create your dedicated schema using the `CREATE SCHEMA IF NOT EXISTS` command

> You only need to replace the ...... placeholders.

In [0]:
load_hint("notebook_1", "step_2")

In [0]:

# 2.1 Create the consulting operations schema
spark.sql(f""" CREATE SCHEMA IF NOT EXISTS `{SCHEMA_NAME}`""")

# 2.2 Set active schema context
spark.sql(f"USE SCHEMA `{SCHEMA_NAME}`")

# Verify complete path context
display(spark.sql("SELECT current_catalog(), current_schema()"))


## Step 3: Create Volume & Manual File Upload

In Unity Catalog, a Volume serves as a managed cloud storage directory for raw files. 

In this step, you will create a Volume, generate the sample data files on your machine (or run a helper cell to create them locally), and manually upload them using the Databricks interface.

Create your dedicated volume using the `CREATE VOLUME IF NOT EXISTS` command.

> You only need to replace the ...... placeholders.

In [0]:
load_hint("notebook_1", "step_3")

In [0]:

# 3.1 Create a Managed Volume for raw landing files
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`{VOLUME_NAME}`")

print(f"Volume Created at: /Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}")


Now that you have created your volume, it's time to upload the given files!
You can find them in the `Raw files` folder.

Some steps to help you with that:

- In the left navigation sidebar of Databricks, click Catalog.
- Navigate to your catalog -> schema -> volume we have created in the previous steps.
- Click Upload to this volume in the top right.
- Drag and drop raw_consultants.csv, raw_projects.parquet and raw_timesheets.json into the volume upload window.

A short description of these files follows:

_**raw_consultants.csv**_ : Contains the official list of active consultants. Because Backbone updates them manually, expect duplicate records, unassigned practices, and invalid seniority titles.

_**raw_timesheets.json**_ : Contains daily logged billable hours (from something like Pepper Time Tracking 🌶️). Mobile feeds are notoriously messy, watch out for corrupt date strings and negative logged hours entered by mistake.

_**raw_projects.parquet**_ : Contains high-level project metadata, budgets, and industry classifications exported from the Project Management Office's database.

💡 Now that we have uploaded the source files via the Databricks interface, let's run a quick check to make sure all three files (raw_consultants.csv, raw_timesheets.json, and raw_projects.parquet) are present and non-empty.

> Run the cells below to inspect the volume content

In [0]:

# 3.2 Inspect files in the raw_landing Volume using DBUtils
volume_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"

print(f"Checking landing path: {volume_path}\n")

landing_files = dbutils.fs.ls(volume_path)
file_names = [f.name for f in landing_files]

# Display directory listing in notebook UI
display(landing_files)

## Step 4: Multi-Source Profiling & Quality Diagnostics

🔎 Since all the files have been uploaded successfully, now it's time to read and inspect them! 

> Execute the cells below to read in the files and take a first look at the data

In [0]:

# 4.1 Read the multi-format source feeds
raw_consultants_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{volume_path}/raw_consultants.csv")
raw_timesheets_df = spark.read.json(f"{volume_path}/raw_timesheets.json")
raw_projects_df = spark.read.parquet(f"{volume_path}/raw_projects.parquet")

print("=== 1. CONSULTANTS (CSV) ===")
display(raw_consultants_df)

print("=== 2. TIMESHEETS (JSON) ===")
display(raw_timesheets_df)

print("=== 3. PROJECTS (PARQUET) ===")
display(raw_projects_df)

🧶 Hmmm legacy exports are notoriously buggy : columns have incorrect data types, primary keys are duplicated, nulls are unhandled, and invalid business logic (like negative billable hours) is present.

Let's clean them before moving on 🧹🧹

### 🧩 Hands-On Task 4.1 (Easy): Identify primary key duplicates in the consultant's list
> **Scenario**: In raw_consultants_df, the primary key is consultant_id. Complete the code below to count how many duplicate rows exist.



In [0]:
load_hint("notebook_1", "step_4.1")

In [0]:

# TASK 1: Fill in the blank to calculate duplicate consultant records
total_consultants = raw_consultants_df.count()
distinct_consultants = raw_consultants_df.dropDuplicates(["consultant_id"]).count()

duplicate_count = total_consultants - distinct_consultants
print(f"Total Consultant Rows: {total_consultants} | Duplicates: {duplicate_count}")


### 🧩 Hands-On Task 4.2 (Medium): Spot Missing & Null Data in the consultant's list

> **Scenario**: Complete the code to scan every column in raw_consultants_df for NULL values.


In [0]:
load_hint("notebook_1", "step_4.2")

In [0]:

# TASK 2: Fill in the blank to count NULLs across all columns
from pyspark.sql.functions import col, count, when

null_summary_df = raw_consultants_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in raw_consultants_df.columns
])

display(null_summary_df)


### 🧩 Hands-On Task 4.3 (Medium): Detect Corrupted Date Formats in Timesheets

> **Scenario**: In raw_timesheets_df, some entries in the work_date column contain corrupt text (e.g., "corrupted_date"). When PySpark attempts to parse an invalid string format into a real date using to_date(), it returns NULL.
Your task is to write a filter that isolates rows where to_date(col("work_date"), "yyyy-MM-dd") returns NULL.

In [0]:
load_hint("notebook_1", "step_4.3")

In [0]:

# TASK 3: Fill in the blank to isolate records with invalid work_date strings
from pyspark.sql.functions import col, try_to_date

invalid_date_df = raw_timesheets_df.filter(
    try_to_date(col("work_date"), "yyyy-MM-dd").isNull()
)
print("Records with Invalid Date Formats:")
display(invalid_date_df)


### 🧩 Hands-On Task 4.4 (Hard): Isolate Multi-Bug Logic Violations

> Scenario: Now, combine your findings! The mobile timesheet application contains two critical data issues:
- Negative or zero logged billable hours (hours <= 0).
- Invalid date strings (where to_date() returns NULL).

Complete the code below using the | (OR) operator to filter out all corrupt timesheet records.

🫠 No hints now, you're on your own! (sorry)

In [0]:

# TASK 4: Fill in the blanks to isolate timesheets with invalid hours OR invalid dates
from pyspark.sql.functions import col, try_to_date

all_corrupt_timesheets = raw_timesheets_df.filter(
     (col("hours") <= 0) | (try_to_date(col("work_date"), "yyyy-MM-dd").isNull())
)

print("All Corrupt Timesheet Records:")
display(all_corrupt_timesheets)


##💡 And now you may ask... why did we do all these things?

Why did we spend all this time setting up Unity Catalog, creating volumes, uploading raw files, and profiling nulls, bad dates, and duplicate keys?

Because now we get down to real business! 
<br>
<br>
<img src="../Images/finally.jpg" width="300">

Without governance and profiling, you're just writing code against a moving target. But now that we know exactly what's broken in our source files, we can build a proper, enterprise-grade Medallion Architecture:

- **Bronze (Raw Ingestion)**: We grab the raw source files as-is and preserve them with audit metadata. No matter what happens downstream, we never lose our raw history.

- **Silver (Data Cleansing & Standardizing)**: This is where we apply the fixes from our profiling tasks—deduplicating primary keys, filtering corrupt dates/negative hours, imputing nulls, and generating cryptographic SHA-256 hash keys for fast joins.

- **Gold (Business Aggregations)**: We combine our pristine Silver tables into clean, high-performance KPI tables ready for business dashboards.

## Step 5: Implement Medallion Architecture (Raw -> Cleaned)

We are ready to build the Medallion Pipeline. We will construct three distinct data layers in PySpark and Spark SQL.

<br>
<pre>
   +-----------------------+      +-----------------------+      +-----------------------+
   |     BRONZE LAYER      | ---> |     SILVER LAYER      | ---> |      GOLD LAYER       |
   | (Raw Ingestion +      |      | (Cleaned, Typed,      |      | (Aggregated KPIs      |
   |  Audit Metadata)      |      |   Hashed Keys)        |      |  for Reporting)       |
   +-----------------------+      +-----------------------+      +-----------------------+
</pre>

### 🥉 5.1 Bronze Layer (Raw Ingestion & Lineage Tracking)

A core data engineering standard is to never modify raw source data on landing. 

In Bronze, we ingest all raw feeds directly into Delta Lake tables while attaching two audit columns: _ingested_at (current timestamp) and _source_file (origin file path).

### 🧩 Hands-On Exercise 5.1: Append Audit Metadata

> Complete the PySpark transformation below to append the system timestamp and file path lineage to the raw consultants DataFrame before saving it as a Delta table.


💡 .write.format("delta") has three modes: 

- _append_: Adds new incoming DataFrame records to the existing Delta table without modifying or deleting existing rows.

- _overwrite_: Completely wipes out the existing Delta table data (or specified partitions) and replaces it with the new incoming DataFrame.

- _ignore_: Silently skips the write operation if the target Delta table already exists, preserving the current data without throwing an error.

- _error_: Throws a runtime exception if the target Delta table already exists, preventing any data from being written.

in this workshop we would like to truncate-replace the target tables, which mode should you use? 🤔

In [0]:
load_hint("notebook_1", "step_5.1")

In [0]:

# 5.1.1 Add lineage metadata to Bronze Consultants
from pyspark.sql.functions import current_timestamp

bronze_consultants_df = (
    raw_consultants_df
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file_path", col("_metadata.file_path"))
)

# Write to Bronze Delta Table
bronze_consultants_df.write.format("delta").mode("overwrite").saveAsTable("bronze_consultants")

# Ingest Timesheets and Projects into Bronze
(raw_timesheets_df
 .withColumn("_ingested_at", current_timestamp())
 .withColumn("_source_file_path", col('_metadata.file_path'))
 .write.format("delta").mode("overwrite").saveAsTable("bronze_timesheets"))

(raw_projects_df
 .withColumn("_ingested_at", current_timestamp())
 .withColumn("_source_file_path", col('_metadata.file_path'))
 .write.format("delta").mode("overwrite").saveAsTable("bronze_projects"))

print("Bronze Layer successfully populated with audit metadata")
display(spark.table("bronze_consultants"))
display(spark.table("bronze_timesheets"))
display(spark.table("bronze_projects"))



👈👈 Now check on the left column of the notebook -> Catalog -> My organization -> {your catalog} -> {your schema} -> Tables and you can see the tables you just created!

(If they don't appear click the Refresh button)

### 🥈 5.2 Silver Layer (Cleansing, Standardization & Hash Keys)
Now we take the raw Bronze data and turn it into a high-grade operational data asset. Based on our Step 4 diagnostics, we must:

- Deduplicate: Eliminate duplicate entries based on natural keys (consultant_id, entry_id).

- Sanitize Nulls & Corrupt Values: Impute default strings for missing attributes and filter out records with invalid dates or negative hours.

- Generate Surrogate Hash Keys: Generate SHA-256 cryptographic hashes (sha2) across natural key attributes.

### 🧩 Hands-On Exercise 5.2: Build the Silver Timesheets Table

> Complete the PySpark pipeline below to cleanse raw timesheet logs and generate a cryptographic hash surrogate key (timesheet_hash_key).

In [0]:
load_hint("notebook_1", "step_5.2")

In [0]:

# 5.2.1 Build Silver Consultants Table
from pyspark.sql.functions import sha2, concat_ws, try_to_date, coalesce, lit, when, col

silver_consultants_df = (
    spark.table("bronze_consultants")
    .dropDuplicates(["consultant_id"])
    .withColumn("practice", coalesce(col("practice"), lit("General Consulting")))
    .withColumn("seniority_level", when(col("seniority_level") == "invalid_level", lit("Unassigned")).otherwise(col("seniority_level")))
    .withColumn("consultant_hash_key", sha2(concat_ws("||", col("consultant_id"), col("full_name")), 256))
)
silver_consultants_df.write.format("delta").mode("overwrite").saveAsTable("silver_consultants")

# 5.2.2 Build Silver Projects Table
silver_projects_df = (
    spark.table("bronze_projects")
    .dropDuplicates(["project_code"])
    .withColumn("project_hash_key", sha2(concat_ws("||", col("project_code"), col("project_name")), 256))
)
silver_projects_df.write.format("delta").mode("overwrite").saveAsTable("silver_projects")

# 5.2.3 EXERCISE: Complete the Silver Timesheets Cleansing Pipeline
silver_timesheets_df = (
    spark.table("bronze_timesheets")
    # Task A: Remove duplicate entries based on 'entry_id'
    .dropDuplicates(["entry_id"])
    # Task B: Keep ONLY valid records (hours > 0 AND work_date is NOT null after parsing)
    .filter((col("hours") > 0) & (try_to_date(col("work_date"), "yyyy-MM-dd").isNotNull()))
    # Task C: Standardize date data type
    .withColumn("work_date", try_to_date(col("work_date"), "yyyy-MM-dd"))
    # Task D: Generate SHA-256 Hash Key on 'entry_id' and 'consultant_id'
    .withColumn("timesheet_hash_key", sha2(concat_ws("||", col("entry_id"), col("consultant_id")), 256))
)

# Save to Silver Delta Table
silver_timesheets_df.write.format("delta").mode("overwrite").saveAsTable("silver_timesheets")

print("✨ Congratulations! Silver Layer is now clean! ✨")

print("Silver consultants table:")
display(spark.table("silver_consultants"))
print("Silver projects table:")
display(spark.table("silver_projects"))
print("Silver timesheets table:")
display(spark.table("silver_timesheets"))


### 🥇 5.3 Gold Layer (Reporting Aggregations)
In Gold, we build lightweight, aggregate tables from our Silver data to answer business questions. In this workshop we will try to reply to the following questions:

> **Query A**: Hours logged per consultant practice group.🕐

> **Query B**: Total project budgets and revenue by industry sector. 💵

### 🧩 Hands-On Task 5.3.1: Practice Hours Summary
_“How many total billable hours did each practice group log?”_ 🤔💭

> Join clean timesheets with the consultant table to calculate total hours logged per practice.

In [0]:
load_hint("notebook_1", "step_5.3.1")

In [0]:

# 5.3.1 Practice Hours Summary Table
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`gold_practice_hours` AS
SELECT 
    c.practice,
    -- Task A: Count unique consultants in each practice
    COUNT(DISTINCT c.consultant_id) AS active_consultants,
    -- Task B: Calculate total logged billable hours
    SUM(t.hours) AS total_logged_hours
FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`silver_timesheets` t
JOIN `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`silver_consultants` c 
    ON t.consultant_id = c.consultant_id
GROUP BY c.practice
ORDER BY total_logged_hours DESC
""")

display(spark.sql(f"SELECT * FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`gold_practice_hours`"))


### 🧩 Hands-On Task 5.3.2: Average Hours per Workday
_““On average, how many hours does a consultant log per work entry across different practices?””_ 🤔💭

> Compute the average workload per logged entry using the timesheets and consultants tables.

In [0]:
load_hint("notebook_1", "step_5.3.2")

In [0]:

# 5.3.2 Average Workload per Entry
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`gold_avg_workload` AS
SELECT 
    c.practice,
    -- Task A: Calculate the average hours logged per entry, rounded to 2 decimal places
    ROUND(AVG(t.hours), 2) AS avg_hours_per_entry
FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`silver_timesheets` t
JOIN `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`silver_consultants` c 
    ON t.consultant_id = c.consultant_id
GROUP BY c.practice
ORDER BY avg_hours_per_entry DESC
""")

display(spark.sql(f"SELECT * FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`gold_avg_workload`"))


%md
### 🧩 Hands-On Task 5.3.3: Project Financial Portfolio Summary

The PMO team needs a breakdown of project finances by industry. Create a Gold table named gold_project_financials that displays:

- industry

- The highest single project budget in that industry (max_budget)

- The average project budget across that industry (avg_budget), rounded to 2 decimal places

Order the final result by max_budget in descending order.

In [0]:

spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`gold_project_financials` AS
SELECT 
    industry,
    MAX(budget) AS max_budget,
    ROUND(AVG(budget), 2) AS avg_budget
FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`silver_projects`
GROUP BY industry
ORDER BY max_budget DESC
""")
display(spark.sql(f"SELECT * FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`gold_project_financials`"))


In [0]:
%sql
--DROP CATALOG IF EXISTS data_eng_alexandra_k CASCADE;